In [2]:
import sqlite3
import pandas as pd

# Setting up the local SQL database for feature engineering
df=pd.read_csv('../data/cleaned_machine_logs.csv')
conn=sqlite3.connect('../data/factory_db.db')
df.to_sql('sensor_logs', conn, if_exists='replace', index=False)

print("--Sql Database and 'sensor_logs' table created successfully---") 



--Sql Database and 'sensor_logs' table created successfully---


In [4]:
# Fetching data back from SQL with LAG() feature
query="""
SELECT 
    timestamp,
    machine_id,
    spindle_speed_rpm,
    vibration_mm_s,
    temperature_c,
    LAG(temperature_c,1) OVER(PARTITION BY machine_id ORDER BY timestamp) AS prev_temperature_c,
    LAG(vibration_mm_s,1) OVER(PARTITION BY machine_id ORDER BY timestamp) AS prev_vibration_mm_s,
    failure
FROM sensor_logs
"""
# Executing the query and pulling the results back into a new Pandas DataFrame
df_features=pd.read_sql_query(query,conn)

print("--- SQL Feature Engineering Applied Successfully---")
display(df_features.head(10))


--- SQL Feature Engineering Applied Successfully---


,timestamp,machine_id,spindle_speed_rpm,vibration_mm_s,temperature_c,prev_temperature_c,prev_vibration_mm_s,failure
0,2026-07-01 08:15:00,CNC_01,11866.131624,1.583399,NaN,NaN,NaN,1
1,2026-07-01 09:00:00,CNC_01,11873.957785,1.358554,63.921504,NaN,1.583399,0
2,2026-07-01 09:15:00,CNC_01,11861.575472,1.133709,68.579311,63.921504,1.358554,0
3,2026-07-01 11:00:00,CNC_01,11735.575127,1.272553,61.572214,68.579311,1.133709,0
4,2026-07-01 11:45:00,CNC_01,12013.099659,1.198834,66.465839,61.572214,1.272553,0
5,2026-07-01 13:00:00,CNC_01,12132.691018,1.336408,66.135506,66.465839,1.198834,0
6,2026-07-01 13:15:00,CNC_01,11862.122979,1.062120,63.231828,66.135506,1.336408,0
7,2026-07-01 14:00:00,CNC_01,11800.026403,1.214285,64.872548,63.231828,1.062120,0
8,2026-07-01 14:15:00,CNC_01,12362.218650,1.201002,57.926924,64.872548,1.214285,0
9,2026-07-01 14:30:00,CNC_01,12011.999007,1.334538,61.359890,57.926924,1.201002,0


In [5]:
df_features_clean=df_features.dropna()
df_features_clean.to_csv('../data/final_features_logs.csv', index=False)
print(f"---Final dataset saved, ready for ML. Total rows:{len(df_features_clean)}---")

---Final dataset saved, ready for ML. Total rows:2996---
